# Load description for each variable in each pair

Recreates and extends analysis from https://github.com/amit-sharma/chatgpt-causality-pairs
Focuses on analysis of the Tübingen dataset from https://webdav.tuebingen.mpg.de/cause-effect/

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
#pip install '/content/drive/MyDrive/pywhy-llm'

In [ ]:
#pip install guidance

In [ ]:
#pip install python-dotenv

In [1]:
import sys
import os
import time


# Add the notebook setup path to sys.path
notebook_setup_path = os.path.abspath("../")
sys.path.insert(0, notebook_setup_path)

from notebook_setup import setup_local_pywhyllm
project_root = setup_local_pywhyllm()



📋 Current sys.path before adding project root:
  0: /home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks
  1: /home/moleropa/miniforge3/envs/tfmenv/lib/python311.zip
  2: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11
  3: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/lib-dynload
  4: 
  5: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/site-packages

🎯 Project root to add: /home/moleropa/repositories/master/TFM/pywhyllm
✅ Added local pywhyllm source to Python path: /home/moleropa/repositories/master/TFM/pywhyllm

📋 Updated sys.path after adding project root:
  0: /home/moleropa/repositories/master/TFM/pywhyllm ⭐
  1: /home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks
  2: /home/moleropa/miniforge3/envs/tfmenv/lib/python311.zip
  3: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11
  4: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/lib-dynload
  5: 
  6: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/site-packages
📋 Current sys.p

In [2]:
from dotenv import load_dotenv
from typing import Dict, List, Tuple
import guidance
import os

from openai import OpenAI
from portkey_ai import createHeaders
import re
import math
from openai import OpenAI

load_dotenv()


True

In [3]:
azure_model= "gpt-4o-mini" #"GPT-4o-2024-05-13" 
us_base_url = "https://us.aigw.galileo.roche.com/v1"

portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])
# azure_openai_client = OpenAI(base_url=us_base_url,
#             api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
#             default_headers=portkey_headers)


# Guidance con modelo OpenAI + base_url + headers
model = guidance.models.OpenAI(
    #"GPT-4o-2024-05-13",
    azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers,
)

azure_model= "gpt-4o-mini" #"GPT-4o-2024-05-13" 
us_base_url = "https://us.aigw.galileo.roche.com/v1"
portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


In [4]:
from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester
modeler= SimpleModelSuggester(llm=model)

In [6]:
import pandas as pd

In [7]:
#df = pd.read_csv('/content/drive/MyDrive/pywhy-llm/pywhyllm/tuebingen_pairs.csv')
df = pd.read_csv('/home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/tuebingen_pairs.csv')

# Get relationship of each variable pair

In [8]:
llm_output : Dict[str, dict] = {}

####  Variables + Straight Strategy

## ALBA TEST
modify these variables to run tests or run everything completely

#TO DO: ready to run everything

In [10]:
# Define parameters for the experiment
temperature = 0.3
num_runs = 1 # Reduced for testing

# Create saved_pairs_info to store ground truth and variable info
saved_pairs_info = {}

# Process only the first 3 rows for testing
#test_df = df.head(1) #or just df all dataset
test_df = df #or just df all dataset
    

In [11]:
from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester
modeler= SimpleModelSuggester(llm=model)

In [12]:
# Iterate through each pair and run multiple times
start = time.time()

for pair_number, values in test_df.iterrows():
    pair_id = f"pair{pair_number:04d}"  # Create pair ID like "pair0001", "pair0002", etc.

    saved_pairs_info[pair_id] = {
        "var1": values['var1'],
        "var2": values['var2'],
        "ground_truth": values['ground_truth'],
    }
    
    for n in range(1, num_runs + 1):
        temp_dict = {}
        
        print(f"Processing {pair_id}, run {n}/{num_runs}")
        
        # Test A → B direction: call suggest_relationships with [var1, var2]
        print(f"  Testing A→B direction: {values['var1']} → {values['var2']}")
        variables_ab = [values['var1'], values['var2']]
        relationships_ab = modeler.suggest_relationships(variables_ab)
        
        # Extract result for A→B
        if not relationships_ab:
            # No relationship found (answer was C)
            temp_dict['llm_ab'] = [None, None, "No causal relationship found"]
        else:
            # Get the relationship - store EXACTLY what LLM said
            (cause, effect), description = list(relationships_ab.items())[0]
            # Always store the actual LLM response without modification
            temp_dict['llm_ab'] = [cause, effect, description]
        
        # Test B → A direction: call suggest_relationships with [var2, var1]
        print(f"  Testing B→A direction: {values['var2']} → {values['var1']}")
        variables_ba = [values['var2'], values['var1']]
        relationships_ba = modeler.suggest_relationships(variables_ba)
        
        # Extract result for B→A
        if not relationships_ba:
            # No relationship found (answer was C)
            temp_dict['llm_ba'] = [None, None, "No causal relationship found"]
        else:
            # Get the relationship - store EXACTLY what LLM said
            (cause, effect), description = list(relationships_ba.items())[0]
            # Always store the actual LLM response without modification
            temp_dict['llm_ba'] = [cause, effect, description]
        
        # Store results with key: (pair_id, temperature, run_number)
        llm_output[(pair_id, temperature, n)] = temp_dict
        
        print(f"  Results stored:")
        print(f"    A→B: {temp_dict['llm_ab'][0]} → {temp_dict['llm_ab'][1]}")
        print(f"    B→A: {temp_dict['llm_ba'][0]} → {temp_dict['llm_ba'][1]}")

# Calculate latencies after all processing is complete
total_time = time.time() - start
print(f"\nCompleted processing {len(test_df)} pairs with {num_runs} runs each")
print(f"Total execution time: {total_time:.2f}s")

# Calculate latencies
avg_latency_per_pair = total_time / (len(test_df) * num_runs)  # Time per pair (both A->B and B->A)
avg_latency_per_run = total_time / (len(test_df) * num_runs * 2)  # Time per individual query

print(f"Average time per pair (A→B + B→A): {avg_latency_per_pair:.2f}s")
print(f"Average time per individual query: {avg_latency_per_run:.2f}s")

Processing pair0000, run 1/1
  Testing A→B direction:  Altitude →  Temperature
1/1.0: Querying for relationship between  Altitude and  Temperature


StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

	 Altitude causes  Temperature
  Testing B→A direction:  Temperature →  Altitude
1/1.0: Querying for relationship between  Temperature and  Altitude
	 Altitude causes  Temperature
  Results stored:
    A→B:  Altitude →  Temperature
    B→A:  Altitude →  Temperature
Processing pair0001, run 1/1
  Testing A→B direction:  Altitude →  Precipitation
1/1.0: Querying for relationship between  Altitude and  Precipitation
	 Altitude causes  Precipitation
  Testing B→A direction:  Precipitation →  Altitude
1/1.0: Querying for relationship between  Precipitation and  Altitude
	 Altitude causes  Precipitation
  Results stored:
    A→B:  Altitude →  Precipitation
    B→A:  Altitude →  Precipitation
Processing pair0002, run 1/1
  Testing A→B direction:  Longitude →  Temperature
1/1.0: Querying for relationship between  Longitude and  Temperature
	No relationship found between  Longitude and  Temperature
  Testing B→A direction:  Temperature →  Longitude
1/1.0: Querying for relationship between  Temp

In [49]:
relationships_ab

{(' Altitude',
  ' Temperature'): "Let's break down the two options:\n\nA. Altitude causes Temperature:  \nAltitude refers to how high something is above sea level. As altitude increases (the higher you go), the temperature generally decreases (it gets colder). This is due to lower air pressure and thinner air at higher elevations, which causes heat to dissipate more rapidly. This is a well-established causal relationship in meteorology and physics.\n\nB. Temperature causes Altitude:  \nThere's no scientific evidence that changes in temperature can directly cause changes in altitude. Temperature changes might affect air density and lead to phenomena like convection, but they do not create or raise land or mountain altitude.\n\nC. Neither Altitude nor Temperature cause each other:  \nThis is not accurate because, as shown above, altitude does affect temperature.\n\nTherefore, the best answer is:\n\n<answer>A</answer>"}

In [56]:
relationships_ba

{(' Altitude',
  ' Temperature'): 'The more likely cause-and-effect relationship is that altitude causes temperature. As altitude increases (i.e., you go higher above sea level), the air becomes thinner and cooler, leading to a drop in temperature. Temperature does not determine altitude; rather, altitude tends to influence temperature due to atmospheric physical properties. Therefore, the effect "Altitude causes Temperature" makes more scientific sense than the reverse or neither.\n\n<answer>B</answer>'}

HALLBAYES

In [13]:
# Create hallbayes backend using the existing Azure OpenAI client
# We need to bypass the default OpenAI API key requirement
import os
from hallbayes import OpenAIBackend, OpenAIItem, OpenAIPlanner
temp_api_key = os.environ.get("PORTKEY_AZURE_US_API_KEY")

# Create the backend with our Azure model and then replace the client
hallbayes_backend = OpenAIBackend(model=azure_model, api_key=temp_api_key)

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


# Replace the default client with our configured Azure OpenAI client 
hallbayes_backend.client = azure_openai_client

planner = OpenAIPlanner(hallbayes_backend, temperature=0.3)

In [14]:
# Validate all A→B and B→A relationships using HallBayes
def validate_all_llm_output_with_hallbayes(llm_output, saved_pairs_info, planner, method="closed_book", threshold=0.10, n_runs=1):
    """
    Validates all causal relationships from llm_output using hallbayes
    
    Args:
        llm_output: dict with keys (pair_id, temperature, run) containing llm_ab and llm_ba results
        saved_pairs_info: dict with pair information (var1, var2, ground_truth)
        planner: configured hallbayes planner
        method: "closed_book" or "with_evidence"
        threshold: max acceptable hallucination risk
        n_runs: number of validation runs per relationship (for consistency)
    
    Returns:
        dict: validation results for each relationship
    """
    print(f"Validating relationships using {method.upper()} method")
    print(f"Threshold: {threshold:.1%}")
    print(f"Validation runs per relationship: {n_runs}")
    print(f"Total pairs to process: {len(llm_output)}")
    print("=" * 60)
    
    all_results = {}
    
    # Process each pair and run combination
    for (pair_id, temp, run_num), output in llm_output.items():
        print(f"\n{'='*60}")
        print(f"Processing {pair_id}, run {run_num}")
        print(f"{'='*60}")
        
        pair_info = saved_pairs_info[pair_id]
        var1, var2 = pair_info['var1'], pair_info['var2']
        ground_truth = pair_info['ground_truth']
        
        # Extract A→B result
        ab_result = output['llm_ab']
        if ab_result[0] is not None and ab_result[1] is not None:
            ab_cause, ab_effect, ab_description = ab_result[0], ab_result[1], ab_result[2]
            
            print(f"\n🔍 Validating A→B: {ab_cause} → {ab_effect}")
            
            # Validate A→B relationship
            ab_validation = _validate_single_relationship(
                ab_cause, ab_effect, ab_description, planner, method, threshold, n_runs
            )
            ab_validation['direction'] = 'A→B'
            ab_validation['var1'] = var1
            ab_validation['var2'] = var2
        else:
            print(f"\n⊘ A→B: No relationship found")
            ab_validation = {
                'direction': 'A→B',
                'var1': var1,
                'var2': var2,
                'final_decision': False,
                'consistency': 'NO_RELATIONSHIP',
                'consistency_rate': 0.0,
                'avg_risk': 1.0,
                'valid_runs': 0,
                'total_runs': 0,
                'description': ab_result[2]
            }
        
        # Extract B→A result
        ba_result = output['llm_ba']
        if ba_result[0] is not None and ba_result[1] is not None:
            ba_cause, ba_effect, ba_description = ba_result[0], ba_result[1], ba_result[2]
            
            print(f"\n🔍 Validating B→A: {ba_cause} → {ba_effect}")
            
            # Validate B→A relationship
            ba_validation = _validate_single_relationship(
                ba_cause, ba_effect, ba_description, planner, method, threshold, n_runs
            )
            ba_validation['direction'] = 'B→A'
            ba_validation['var1'] = var2
            ba_validation['var2'] = var1
        else:
            print(f"\n⊘ B→A: No relationship found")
            ba_validation = {
                'direction': 'B→A',
                'var1': var2,
                'var2': var1,
                'final_decision': False,
                'consistency': 'NO_RELATIONSHIP',
                'consistency_rate': 0.0,
                'avg_risk': 1.0,
                'valid_runs': 0,
                'total_runs': 0,
                'description': ba_result[2]
            }
        
        # Store results
        key = (pair_id, temp, run_num)
        all_results[key] = {
            'pair_id': pair_id,
            'ground_truth': ground_truth,
            'ab_validation': ab_validation,
            'ba_validation': ba_validation
        }
        
        # Print summary
        print(f"\n{'─'*60}")
        print(f"📊 Summary for {pair_id}, run {run_num}:")
        print(f"  Ground Truth: {ground_truth}")
        print(f"  A→B ({var1} → {var2}): {'✅ VALID' if ab_validation['final_decision'] else '❌ REJECT'} " + 
              f"| Consistency: {ab_validation['consistency']} ({ab_validation['valid_runs']}/{ab_validation['total_runs']})")
        print(f"  B→A ({var2} → {var1}): {'✅ VALID' if ba_validation['final_decision'] else '❌ REJECT'} " + 
              f"| Consistency: {ba_validation['consistency']} ({ba_validation['valid_runs']}/{ba_validation['total_runs']})")
    
    return all_results


def _validate_single_relationship(cause, effect, description, planner, method, threshold, n_runs):
    """
    Validates a single causal relationship using hallbayes with consistency checking
    """
    # Prepare prompt based on method
    if method == "closed_book":
        prompt = f"""
Causal Knowledge Assessment:

Claim: "{cause}" causes "{effect}"

Question: Based on established scientific knowledge, 
is this causal relationship scientifically valid?

Answer: Yes/No with brief justification.
"""
    elif method == "with_evidence":
        prompt = f"""
Evidence-Based Assessment:

Provided Evidence: {description}

Claim: "{cause}" causes "{effect}"

Question: Based on the provided evidence and your knowledge, 
is this causal relationship valid and well-supported?

Answer: Yes/No referencing the evidence and additional knowledge.
"""
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Run multiple validations for consistency
    run_results = []
    valid_count = 0
    
    for run in range(n_runs):
        try:
            item = OpenAIItem(prompt=prompt, n_samples=3, m=4, skeleton_policy="closed_book")
            print(f"    Run {run + 1}/{n_runs}: Generating HallBayes samples...")
            metrics = planner.run([item], h_star=threshold, isr_threshold=1.0)
            
            if metrics:
                metric = metrics[0]
                is_valid = metric.decision_answer
                risk = metric.roh_bound
                
                if is_valid:
                    valid_count += 1
                
                run_results.append({
                    'valid': is_valid,
                    'risk': risk,
                    'run': run + 1
                })
                
                status = "✅ VALID" if is_valid else "❌ REJECT"
                print(f"    Run {run + 1}: {status} (Risk: {risk:.1%})")
            else:
                run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': 'No metrics'})
                print(f"    Run {run + 1}: ⚠️  ERROR (No metrics)")
                
        except Exception as e:
            run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': str(e)})
            print(f"    Run {run + 1}: ⚠️  ERROR ({str(e)[:50]}...)")
    
    # Calculate consistency and final decision
    consistency_rate = valid_count / n_runs if n_runs > 0 else 0.0
    avg_risk = sum(r.get('risk', 1.0) for r in run_results) / len(run_results) if run_results else 1.0
    
    # Determine final validation (majority vote)
    final_valid = valid_count > (n_runs / 2) if n_runs > 0 else False
    
    # Determine consistency level
    if consistency_rate == 1.0 or consistency_rate == 0.0:
        consistency = "CONSISTENT"
    elif consistency_rate >= 0.66:
        consistency = "MOSTLY_CONSISTENT"
    else:
        consistency = "INCONSISTENT"
    
    return {
        'final_decision': final_valid,
        'consistency': consistency,
        'consistency_rate': consistency_rate,
        'avg_risk': avg_risk,
        'valid_runs': valid_count,
        'total_runs': n_runs,
        'method': method,
        'runs': run_results,
        'description': description
    }

In [16]:
# Run HallBayes validation on all relationships
print("🚀 Starting HallBayes validation for all pairs...")
print(f"Total pairs to validate: {len(llm_output)}")
print()

hallbayes_validation_results = validate_all_llm_output_with_hallbayes(
    llm_output, 
    saved_pairs_info, 
    planner, 
    method="closed_book",  # Use closed_book since baseline has no RAG
    threshold=0.10,  # 10% max hallucination risk
    n_runs=1  # Number of validation runs per relationship
)

print("\n" + "="*60)
print("✅ HallBayes validation completed!")
print(f"Total validations performed: {len(hallbayes_validation_results)}")
print("="*60)

🚀 Starting HallBayes validation for all pairs...
Total pairs to validate: 108

Validating relationships using CLOSED_BOOK method
Threshold: 10.0%
Validation runs per relationship: 1
Total pairs to process: 108

Processing pair0000, run 1

🔍 Validating A→B:  Altitude →  Temperature
    Run 1/1: Generating HallBayes samples...
    Run 1: ✅ VALID (Risk: 0.0%)

🔍 Validating B→A:  Altitude →  Temperature
    Run 1/1: Generating HallBayes samples...
    Run 1: ✅ VALID (Risk: 0.0%)

────────────────────────────────────────────────────────────
📊 Summary for pair0000, run 1:
  Ground Truth:  R
  A→B ( Altitude →  Temperature): ✅ VALID | Consistency: CONSISTENT (1/1)
  B→A ( Temperature →  Altitude): ✅ VALID | Consistency: CONSISTENT (1/1)

Processing pair0001, run 1

🔍 Validating A→B:  Altitude →  Precipitation
    Run 1/1: Generating HallBayes samples...
    Run 1: ✅ VALID (Risk: 0.0%)

🔍 Validating B→A:  Altitude →  Precipitation
    Run 1/1: Generating HallBayes samples...
    Run 1: ✅ VALID

In [15]:
llm_output

{('pair0000',
  0.3,
  1): {'llm_ab': [' Altitude',
   ' Temperature',
   "To answer this, let's analyze each option:\n\nA. Altitude causes Temperature  \nIt is well established in atmospheric science that temperature generally decreases with altitude in the troposphere (the lowest layer of Earth's atmosphere, where most weather occurs). This is because atmospheric pressure decreases with altitude, resulting in cooler temperatures. Thus, it is the *change in altitude* that leads to changes in temperature, not the other way around.\n\nB. Temperature causes Altitude  \nThere is no natural mechanism where temperature directly causes changes in altitude. While temperature can cause expansion or contraction of air masses and affect things like local air pressure, it does not cause a physical change in the altitude of a place.\n\nC. neither Altitude nor Temperature cause each other.  \nThis would be incorrect, given that physical principles underpin the relationship between altitude and temp

In [17]:
# Calculate comprehensive results including accuracy, latency, and HallBayes validation
results: Dict = {}

for pair_id in saved_pairs_info:
    av_correct_ab = 0
    av_correct_ba = 0
    
    # Lists to collect HallBayes metrics across runs
    hallbayes_ab_risks = []
    hallbayes_ba_risks = []
    hallbayes_ab_valid = []
    hallbayes_ba_valid = []
    hallbayes_ab_consistency = []
    hallbayes_ba_consistency = []

    for i in range(num_runs):
        # Extract LLM relationship results
        ab_result = llm_output[(pair_id, 0.3, i+1)]['llm_ab']
        ba_result = llm_output[(pair_id, 0.3, i+1)]['llm_ba']
        
        # Extract HallBayes validation results
        hallbayes_result = hallbayes_validation_results[(pair_id, 0.3, i+1)]
        ab_validation = hallbayes_result['ab_validation']
        ba_validation = hallbayes_result['ba_validation']
        
        # Store HallBayes metrics
        hallbayes_ab_risks.append(ab_validation['avg_risk'])
        hallbayes_ba_risks.append(ba_validation['avg_risk'])
        hallbayes_ab_valid.append(1 if ab_validation['final_decision'] else 0)
        hallbayes_ba_valid.append(1 if ba_validation['final_decision'] else 0)
        hallbayes_ab_consistency.append(ab_validation['consistency'])
        hallbayes_ba_consistency.append(ba_validation['consistency'])
        
        # Determine the LLM's causal direction prediction
        # ab_result is [cause, effect, description]
        if ab_result[0] is not None and ab_result[1] is not None:
            # Check if A→B (var1 causes var2)
            if ab_result[0] == saved_pairs_info[pair_id]['var1'] and ab_result[1] == saved_pairs_info[pair_id]['var2']:
                ab_relationship = 1  # A causes B
            # Check if B→A (var2 causes var1) 
            elif ab_result[0] == saved_pairs_info[pair_id]['var2'] and ab_result[1] == saved_pairs_info[pair_id]['var1']:
                ab_relationship = 0  # B causes A
            else:
                ab_relationship = None  # Unclear
        else:
            ab_relationship = None  # No relationship found
            
        if ba_result[0] is not None and ba_result[1] is not None:
            # For B→A query: if LLM says var2→var1 (B→A), that's saying the OPPOSITE of var1→var2
            # So we need to check what direction the LLM is claiming
            if ba_result[0] == saved_pairs_info[pair_id]['var2'] and ba_result[1] == saved_pairs_info[pair_id]['var1']:
                # LLM says B→A (var2→var1), which contradicts ground truth " R" (var1→var2)
                ba_relationship = 0  # Claiming B causes A
            elif ba_result[0] == saved_pairs_info[pair_id]['var1'] and ba_result[1] == saved_pairs_info[pair_id]['var2']:
                # LLM says A→B (var1→var2), which aligns with ground truth " R"
                ba_relationship = 1  # Claiming A causes B
            else:
                ba_relationship = None  # Unclear
        else:
            ba_relationship = None  # No relationship found

        # Check correctness against ground truth
        # Ground truth: " R" means var1→var2 (A→B), " L" means var2→var1 (B→A)
        
        # For A→B query: 
        # - If LLM says A→B (ab_relationship=1) and ground truth is " R" (A→B), it's correct
        # - If LLM says B→A (ab_relationship=0) and ground truth is " L" (B→A), it's correct
        if ab_relationship == 1 and saved_pairs_info[pair_id]['ground_truth'] == " R":
            av_correct_ab += 1
        elif ab_relationship == 0 and saved_pairs_info[pair_id]['ground_truth'] == " L":
            av_correct_ab += 1

        # For B→A query:
        # - If LLM says A→B (ba_relationship=1) and ground truth is " R" (A→B), it's correct
        # - If LLM says B→A (ba_relationship=0) and ground truth is " L" (B→A), it's correct
        if ba_relationship == 1 and saved_pairs_info[pair_id]['ground_truth'] == " R":
            av_correct_ba += 1
        elif ba_relationship == 0 and saved_pairs_info[pair_id]['ground_truth'] == " L":
            av_correct_ba += 1

    # Calculate averages
    av_correct_ab /= num_runs
    av_correct_ba /= num_runs
    
    # Calculate average HallBayes metrics
    avg_hallbayes_ab_risk = sum(hallbayes_ab_risks) / len(hallbayes_ab_risks) if hallbayes_ab_risks else None
    avg_hallbayes_ba_risk = sum(hallbayes_ba_risks) / len(hallbayes_ba_risks) if hallbayes_ba_risks else None
    avg_hallbayes_ab_valid_rate = sum(hallbayes_ab_valid) / len(hallbayes_ab_valid) if hallbayes_ab_valid else 0
    avg_hallbayes_ba_valid_rate = sum(hallbayes_ba_valid) / len(hallbayes_ba_valid) if hallbayes_ba_valid else 0
    
    # Get most common consistency level
    from collections import Counter
    ab_consistency_mode = Counter(hallbayes_ab_consistency).most_common(1)[0][0] if hallbayes_ab_consistency else "N/A"
    ba_consistency_mode = Counter(hallbayes_ba_consistency).most_common(1)[0][0] if hallbayes_ba_consistency else "N/A"

    # Store results
    temp: Dict = {}
    temp['PairID'] = pair_id
    temp['VarA'] = saved_pairs_info[pair_id]['var1']
    temp['VarB'] = saved_pairs_info[pair_id]['var2']
    temp['GroundTruth'] = saved_pairs_info[pair_id]['ground_truth']
    temp['AccuracyAB'] = av_correct_ab
    temp['AccuracyBA'] = av_correct_ba
    temp['HallBayesRiskAB'] = avg_hallbayes_ab_risk
    temp['HallBayesRiskBA'] = avg_hallbayes_ba_risk
    temp['HallBayesValidRateAB'] = avg_hallbayes_ab_valid_rate
    temp['HallBayesValidRateBA'] = avg_hallbayes_ba_valid_rate
    temp['HallBayesConsistencyAB'] = ab_consistency_mode
    temp['HallBayesConsistencyBA'] = ba_consistency_mode

    results[pair_id] = temp
    print(f"{pair_id}: Accuracy AB={av_correct_ab:.2f}, BA={av_correct_ba:.2f} | "
          f"HallBayes Risk AB={avg_hallbayes_ab_risk:.1%}, BA={avg_hallbayes_ba_risk:.1%}")

print(f"\n✅ Results calculated for {len(results)} pairs")

pair0000: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0001: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0002: Accuracy AB=0.00, BA=0.00 | HallBayes Risk AB=100.0%, BA=100.0%
pair0003: Accuracy AB=0.00, BA=0.00 | HallBayes Risk AB=100.0%, BA=100.0%
pair0004: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0005: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0006: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=54.1%, BA=54.1%
pair0007: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0008: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0009: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0010: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0011: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0012: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0013: Accuracy AB=1.00, BA=1.00 | HallBayes Risk AB=0.0%, BA=0.0%
pair0014: 

In [20]:
# Calculate overall statistics including accuracy, latency, and HallBayes metrics
print("=" * 80)
print("📊 COMPREHENSIVE STATISTICS: Accuracy + Latency + HallBayes")
print("=" * 80)

total_pairs = len(results)
sum_ab_accuracy = 0
sum_ba_accuracy = 0
sum_joint_accuracy = 0
sum_ab_risk = 0
sum_ba_risk = 0
sum_ab_valid_rate = 0
sum_ba_valid_rate = 0
count_ab_risk = 0
count_ba_risk = 0

for pair_id, result in results.items():
    # Accuracy
    correct_ab = result['AccuracyAB']
    correct_ba = result['AccuracyBA']
    joint_accuracy = (correct_ab + correct_ba) / 2.0
    
    sum_ab_accuracy += correct_ab
    sum_ba_accuracy += correct_ba
    sum_joint_accuracy += joint_accuracy
    
    # HallBayes risk
    if result['HallBayesRiskAB'] is not None:
        sum_ab_risk += result['HallBayesRiskAB']
        count_ab_risk += 1
    if result['HallBayesRiskBA'] is not None:
        sum_ba_risk += result['HallBayesRiskBA']
        count_ba_risk += 1
    
    # HallBayes validation rate
    sum_ab_valid_rate += result['HallBayesValidRateAB']
    sum_ba_valid_rate += result['HallBayesValidRateBA']

# Calculate overall metrics
overall_ab_accuracy = sum_ab_accuracy / total_pairs
overall_ba_accuracy = sum_ba_accuracy / total_pairs
overall_joint_accuracy = sum_joint_accuracy / total_pairs

overall_ab_risk = sum_ab_risk / count_ab_risk if count_ab_risk > 0 else None
overall_ba_risk = sum_ba_risk / count_ba_risk if count_ba_risk > 0 else None

overall_ab_valid_rate = sum_ab_valid_rate / total_pairs
overall_ba_valid_rate = sum_ba_valid_rate / total_pairs

print(f"\n{'Metric':<35} {'A→B':<20} {'B→A':<20}")
print("-" * 80)
print(f"{'Mean Accuracy':<35} {overall_ab_accuracy:<20.3f} {overall_ba_accuracy:<20.3f}")
print(f"{'Joint Mean Accuracy':<35} {overall_joint_accuracy:<20.3f}")
print(f"{'Mean HallBayes Risk':<35} {f'{overall_ab_risk:.1%}' if overall_ab_risk else 'N/A':<20} {f'{overall_ba_risk:.1%}' if overall_ba_risk else 'N/A':<20}")
print(f"{'Mean HallBayes Valid Rate':<35} {overall_ab_valid_rate:<20.3f} {overall_ba_valid_rate:<20.3f}")
print(f"{'Total Pairs':<35} {total_pairs:<20}")

print(f"\n{'LATENCY METRICS':<35}")
print("-" * 80)
print(f"{'Avg time per pair (A→B + B→A)':<35} {avg_latency_per_pair:<20.2f}s")
print(f"{'Avg time per query':<35} {avg_latency_per_run:<20.2f}s")
print(f"{'Total execution time':<35} {total_time:<20.2f}s")

# Create summary dictionary for export
accuracy_hallbayes_summary = {
    'total_pairs': total_pairs,
    'ab_mean_accuracy': overall_ab_accuracy,
    'ba_mean_accuracy': overall_ba_accuracy,
    'joint_mean_accuracy': overall_joint_accuracy,
    'ab_mean_hallbayes_risk': overall_ab_risk,
    'ba_mean_hallbayes_risk': overall_ba_risk,
    'ab_mean_hallbayes_valid_rate': overall_ab_valid_rate,
    'ba_mean_hallbayes_valid_rate': overall_ba_valid_rate,
    'avg_latency_per_pair_seconds': round(avg_latency_per_pair, 2),
    'avg_latency_per_query_seconds': round(avg_latency_per_run, 2),
    'total_execution_time_seconds': round(total_time, 2)
}

print("\n✅ Summary statistics calculated")
print("=" * 80)

📊 COMPREHENSIVE STATISTICS: Accuracy + Latency + HallBayes

Metric                              A→B                  B→A                 
--------------------------------------------------------------------------------
Mean Accuracy                       0.843                0.852               
Joint Mean Accuracy                 0.847               
Mean HallBayes Risk                 29.1%                27.9%               
Mean HallBayes Valid Rate           0.806                0.824               
Total Pairs                         108                 

LATENCY METRICS                    
--------------------------------------------------------------------------------
Avg time per pair (A→B + B→A)       5.24                s
Avg time per query                  2.62                s
Total execution time                565.92              s

✅ Summary statistics calculated


In [22]:
# Export results to CSV (Accuracy + HallBayes metrics)
import csv

# CSV file for detailed results (includes HallBayes metrics)
detailed_csv_file = "baseline_hallbayes_detailed_results_all_pairs.csv"

# Define headers for detailed results
detailed_header = [
    "PairID", "VarA", "VarB", "GroundTruth", 
    "AccuracyAB", "AccuracyBA",
    "HallBayesRiskAB", "HallBayesRiskBA",
    "HallBayesValidRateAB", "HallBayesValidRateBA",
    "HallBayesConsistencyAB", "HallBayesConsistencyBA"
]

# Write detailed results
with open(detailed_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=detailed_header)
    writer.writeheader()
    for pair_id, values in results.items():
        writer.writerow(values)

print(f"✅ Detailed results CSV file '{detailed_csv_file}' has been created.")

# CSV file for summary statistics
summary_csv_file = "baseline_hallbayes_summary_all_pairs.csv"

# Write summary statistics
with open(summary_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(accuracy_hallbayes_summary.keys()))
    writer.writeheader()
    writer.writerow(accuracy_hallbayes_summary)

print(f"✅ Summary CSV file '{summary_csv_file}' has been created.")

# Display final summary table
print("\n" + "=" * 80)
print("📋 FINAL RESULTS SUMMARY")
print("=" * 80)
print(f"\n{'Metric':<40} {'Value':<20}")
print("-" * 80)
print(f"{'Total Pairs Processed':<40} {accuracy_hallbayes_summary['total_pairs']}")
print(f"{'A→B Mean Accuracy':<40} {accuracy_hallbayes_summary['ab_mean_accuracy']:.3f}")
print(f"{'B→A Mean Accuracy':<40} {accuracy_hallbayes_summary['ba_mean_accuracy']:.3f}")
print(f"{'Joint Mean Accuracy':<40} {accuracy_hallbayes_summary['joint_mean_accuracy']:.3f}")
print(f"{'A→B Mean HallBayes Valid Rate':<40} {accuracy_hallbayes_summary['ab_mean_hallbayes_valid_rate']:.3f}")
print(f"{'B→A Mean HallBayes Valid Rate':<40} {accuracy_hallbayes_summary['ba_mean_hallbayes_valid_rate']:.3f}")
print(f"{'Avg Latency per Pair':<40} {accuracy_hallbayes_summary['avg_latency_per_pair_seconds']:.2f}s")
print(f"{'Avg Latency per Query':<40} {accuracy_hallbayes_summary['avg_latency_per_query_seconds']:.2f}s")
print(f"{'Total Execution Time':<40} {accuracy_hallbayes_summary['total_execution_time_seconds']:.2f}s")
print("=" * 80)

print("\n🎉 All results exported successfully!")

✅ Detailed results CSV file 'baseline_hallbayes_detailed_results_all_pairs.csv' has been created.
✅ Summary CSV file 'baseline_hallbayes_summary_all_pairs.csv' has been created.

📋 FINAL RESULTS SUMMARY

Metric                                   Value               
--------------------------------------------------------------------------------
Total Pairs Processed                    108
A→B Mean Accuracy                        0.843
B→A Mean Accuracy                        0.852
Joint Mean Accuracy                      0.847
A→B Mean HallBayes Valid Rate            0.806
B→A Mean HallBayes Valid Rate            0.824
Avg Latency per Pair                     5.24s
Avg Latency per Query                    2.62s
Total Execution Time                     565.92s

🎉 All results exported successfully!


In [12]:
# # Bind the HallBayes function to the modeler instance
# from types import MethodType
# modeler.suggest_pairwise_relationship_with_hallbayes = MethodType(
#     suggest_pairwise_relationship_with_hallbayes, modeler
# )

In [14]:
# # Example usage with HallBayes
# # Note: You must have a planner configured (already done in cell above)

# # Test with a single pair
# test_result = modeler.suggest_pairwise_relationship_with_hallbayes(
#     variable1="hypertension",
#     variable2="Age",
#     planner=planner,
#     h_star=0.05,  # Target max 5% hallucination
#     n_samples=7,   # More samples for stability
#     m=6,           # Number of skeleton variants
#     temperature=0.3
# )


=== HallBayes Causal Inference Result ===
Answer: REFUSED
Decision Approved: False
Hallucination Risk Bound: 0.0000
Information Sufficiency Ratio (ISR): 0.0000
Information Budget (Δ̄): 0.0000 nats
Bits-to-Trust (B2T): 1.1830 nats
Rationale: Δ̄=0.0000 nats, B2T=1.1830, ISR=0.000 (thr=1.000), extra_bits=0.200; EDFL RoH bound=0.000; y='answer'

Full Response:
REFUSED: Δ̄=0.0000 nats, B2T=1.1830, ISR=0.000 (thr=1.000), extra_bits=0.200; EDFL RoH bound=0.000; y='answer'


In [ ]:
# # Run full experiment with HallBayes
# llm_output_hallbayes : Dict[str, dict] = {}
# hallbayes_refusals : Dict[str, dict] = {}

# # Iterate through each pair and run with HallBayes
# start = time.time()

# for pair_number, values in test_df.iterrows():
#     pair_id = f"pair{pair_number:04d}"
    
#     saved_pairs_info[pair_id] = {
#         "var1": values['var1'],
#         "var2": values['var2'],
#         "ground_truth": values['ground_truth'],
#     }
    
#     for n in range(1, num_runs + 1):
#         temp_dict = {}
        
#         print(f"Processing {pair_id}, run {n}/{num_runs} with HallBayes")
        
#         # Test A -> B direction with HallBayes
#         try:
#             temp_dict['llm_ab'] = modeler.suggest_pairwise_relationship_with_hallbayes(
#                 variable1=values['var1'], 
#                 variable2=values['var2'], 
#                 planner=planner,
#                 h_star=0.05,  # 5% max hallucination rate
#                 n_samples=7,
#                 m=6,
#                 temperature=0.3
#             )
#         except Exception as e:
#             print(f"  Error in A->B: {e}")
#             temp_dict['llm_ab'] = {
#                 'answer': 'ERROR',
#                 'decision_answer': False,
#                 'description': str(e)
#             }

#         # Test B -> A direction with HallBayes
#         try:
#             temp_dict['llm_ba'] = modeler.suggest_pairwise_relationship_with_hallbayes(
#                 variable1=values['var2'],
#                 variable2=values['var1'],
#                 planner=planner,
#                 h_star=0.05,
#                 n_samples=7,
#                 m=6,
#                 temperature=0.3
#             )
#         except Exception as e:
#             print(f"  Error in B->A: {e}")
#             temp_dict['llm_ba'] = {
#                 'answer': 'ERROR',
#                 'decision_answer': False,
#                 'description': str(e)
#             }
        
#         # Store results
#         llm_output_hallbayes[(pair_id, temperature, n)] = temp_dict
        
#         # Track refusals
#         if not temp_dict['llm_ab'].get('decision_answer', False):
#             hallbayes_refusals[f"{pair_id}_ab_run{n}"] = temp_dict['llm_ab']
#         if not temp_dict['llm_ba'].get('decision_answer', False):
#             hallbayes_refusals[f"{pair_id}_ba_run{n}"] = temp_dict['llm_ba']
        
#         print(f"  A->B: {temp_dict['llm_ab']['answer']} (Approved: {temp_dict['llm_ab'].get('decision_answer', False)}, " +
#               f"Risk: {temp_dict['llm_ab'].get('hallucination_risk', 'N/A'):.4f})")
#         print(f"  B->A: {temp_dict['llm_ba']['answer']} (Approved: {temp_dict['llm_ba'].get('decision_answer', False)}, " +
#               f"Risk: {temp_dict['llm_ba'].get('hallucination_risk', 'N/A'):.4f})")

# # Calculate execution metrics
# total_time = time.time() - start
# print(f"\n=== HallBayes Execution Complete ===")
# print(f"Total execution time: {total_time:.2f}s")
# print(f"Total refusals: {len(hallbayes_refusals)}")
# print(f"Refusal rate: {len(hallbayes_refusals) / (len(test_df) * num_runs * 2) * 100:.1f}%")

# avg_latency_per_pair = total_time / (len(test_df) * num_runs)
# avg_latency_per_query = total_time / (len(test_df) * num_runs * 2)
# print(f"Average time per pair (A->B + B->A): {avg_latency_per_pair:.2f}s")
# print(f"Average time per query: {avg_latency_per_query:.2f}s")

In [ ]:
results : Dict = {}

for id in saved_pairs_info:

    av_correct_ab = 0
    av_correct_ba = 0
    
    # Lists to collect confidence, strength scores, and references across runs
    confidence_scores_ab = []
    confidence_scores_ba = []
    strength_scores_ab = []
    strength_scores_ba = []
    references_ab = []
    references_ba = []

    for i in range(num_runs):
        # Extract relationship, confidence, strength, and other data from dictionary
        ab_result = llm_output[(id, 0.3, i+1)]['llm_ab']
        ba_result = llm_output[(id, 0.3, i+1)]['llm_ba']
        
        # Get data from result dictionary
        ab_answer = ab_result.get('answer', '')
        ba_answer = ba_result.get('answer', '')
        ab_confidence = ab_result.get('confidence_score')
        ba_confidence = ba_result.get('confidence_score')
        ab_strength = ab_result.get('strength_score')
        ba_strength = ba_result.get('strength_score')
        
        # Store confidence scores (only if not None)
        if ab_confidence is not None:
            confidence_scores_ab.append(ab_confidence)
        if ba_confidence is not None:
            confidence_scores_ba.append(ba_confidence)
        
        # Store strength scores (only if not None)
        if ab_strength is not None:
            strength_scores_ab.append(ab_strength)
        if ba_strength is not None:
            strength_scores_ba.append(ba_strength)
            
        # For baseline method, we don't have references, so use empty lists
        references_ab.extend([])
        references_ba.extend([])

        # Convert answer letters to relationship values for correctness checking
        # A means var1 -> var2 (relationship = 1), B means var2 -> var1 (relationship = 0), C means no relationship
        ab_relationship = 1 if ab_answer == "A" else 0 if ab_answer == "B" else None
        ba_relationship = 1 if ba_answer == "A" else 0 if ba_answer == "B" else None

        # Check correctness using the relationship value
        if ab_relationship == 1 and saved_pairs_info[id]['ground_truth'] == " R":
            av_correct_ab += 1
        elif ab_relationship == 0 and saved_pairs_info[id]['ground_truth'] == " L":
            av_correct_ab += 1

        if ba_relationship == 1 and saved_pairs_info[id]['ground_truth'] == " L":
            av_correct_ba += 1
        elif ba_relationship == 0 and saved_pairs_info[id]['ground_truth'] == " R":
            av_correct_ba += 1

    av_correct_ab /= num_runs
    av_correct_ba /= num_runs
    
    # Calculate average confidence scores
    avg_confidence_ab = sum(confidence_scores_ab) / len(confidence_scores_ab) if confidence_scores_ab else None
    avg_confidence_ba = sum(confidence_scores_ba) / len(confidence_scores_ba) if confidence_scores_ba else None
    
    # Calculate average strength scores
    avg_strength_ab = sum(strength_scores_ab) / len(strength_scores_ab) if strength_scores_ab else None
    avg_strength_ba = sum(strength_scores_ba) / len(strength_scores_ba) if strength_scores_ba else None

    temp : Dict = {}

    temp['PairID'] = id
    temp['CorrectACauseB'] = av_correct_ab
    temp['CorrectBCauseA'] = av_correct_ba
    temp['VarA'] = saved_pairs_info[id]['var1']
    temp['VarB'] = saved_pairs_info[id]['var2']
    temp['GroundTruth'] = saved_pairs_info[id]['ground_truth']
    temp['ConfidenceAB'] = avg_confidence_ab
    temp['ConfidenceBA'] = avg_confidence_ba
    temp['StrengthAB'] = avg_strength_ab  # NEW: Average strength score for A->B
    temp['StrengthBA'] = avg_strength_ba  # NEW: Average strength score for B->A
    temp['ReferencesAB'] = references_ab
    temp['ReferencesBA'] = references_ba

    results[id] = temp
    print(results[id])


{'PairID': 'pair0000', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Altitude', 'VarB': ' Temperature', 'GroundTruth': ' R', 'ConfidenceAB': 0.95, 'ConfidenceBA': 0.95, 'StrengthAB': 0.8, 'StrengthBA': 0.8, 'ReferencesAB': [], 'ReferencesBA': []}
{'PairID': 'pair0001', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Altitude', 'VarB': ' Precipitation', 'GroundTruth': ' R', 'ConfidenceAB': 0.95, 'ConfidenceBA': 0.95, 'StrengthAB': 0.7, 'StrengthBA': 0.7, 'ReferencesAB': [], 'ReferencesBA': []}
{'PairID': 'pair0002', 'CorrectACauseB': 0.0, 'CorrectBCauseA': 0.0, 'VarA': ' Longitude', 'VarB': ' Temperature', 'GroundTruth': ' R', 'ConfidenceAB': 0.95, 'ConfidenceBA': 1.0, 'StrengthAB': 0.0, 'StrengthBA': 0.0, 'ReferencesAB': [], 'ReferencesBA': []}
{'PairID': 'pair0003', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Altitude', 'VarB': ' Sunshine hours', 'GroundTruth': ' R', 'ConfidenceAB': 0.8, 'ConfidenceBA': 0.85, 'StrengthAB': 0.4, 'StrengthBA': 0.4, 'Referen

In [101]:
# Calculate accuracy metrics
accuracy_results = {}

total_pairs = len(results)
sum_ab_accuracy = 0
sum_ba_accuracy = 0
sum_joint_accuracy = 0
sum_ab_confidence = 0
sum_ba_confidence = 0
sum_ab_strength = 0
sum_ba_strength = 0
count_ab_confidence = 0
count_ba_confidence = 0
count_ab_strength = 0
count_ba_strength = 0


for pair_id, result in results.items():
    # Individual accuracies per pair (these are already averages from multiple runs, can be decimal)
    correct_ab = result['CorrectACauseB']  # This is the mean accuracy (0.0 to 1.0, can be decimal)
    correct_ba = result['CorrectBCauseA']  # This is the mean accuracy (0.0 to 1.0, can be decimal)
    
    # Get confidence scores
    confidence_ab = result.get('ConfidenceAB')
    confidence_ba = result.get('ConfidenceBA')
    
    # Get strength scores
    strength_ab = result.get('StrengthAB')
    strength_ba = result.get('StrengthBA')
    
    # Joint accuracy: average of both directions (more nuanced approach)
    joint_accuracy = (correct_ab + correct_ba) / 2.0
    
    # Sum for overall statistics (averaging across all pairs)
    sum_ab_accuracy += correct_ab
    sum_ba_accuracy += correct_ba
    sum_joint_accuracy += joint_accuracy
    
    # Sum confidence scores for overall statistics
    if confidence_ab is not None:
        sum_ab_confidence += confidence_ab
        count_ab_confidence += 1
    if confidence_ba is not None:
        sum_ba_confidence += confidence_ba
        count_ba_confidence += 1
    
    # Sum strength scores for overall statistics
    if strength_ab is not None:
        sum_ab_strength += strength_ab
        count_ab_strength += 1
    if strength_ba is not None:
        sum_ba_strength += strength_ba
        count_ba_strength += 1
    
    # STORING INDIVIDUAL
    # Store individual pair results (only decimal values, no redundant percentages)
    accuracy_results[pair_id] = {
        'PairID': pair_id,
        'VarA': result['VarA'],
        'VarB': result['VarB'],
        'GroundTruth': result['GroundTruth'],
        'AccuracyAB': correct_ab,  # Decimal value (0.0 to 1.0)
        'AccuracyBA': correct_ba,  # Decimal value (0.0 to 1.0)
        'JointAccuracy': joint_accuracy,  # Average of both directions
        'ConfidenceAB': confidence_ab,  # Confidence score for A->B
        'ConfidenceBA': confidence_ba,  # Confidence score for B->A
        'StrengthAB': strength_ab,  # Strength score for A->B
        'StrengthBA': strength_ba   # Strength score for B->A
    }
    
    print(f"Pair {pair_id}: {result['VarA']} -> {result['VarB']}")
    print(f"  Ground Truth: {result['GroundTruth']}")
    print(f"  A→B Accuracy: {correct_ab:.3f}, Confidence: {f'{confidence_ab:.3f}' if confidence_ab is not None else 'N/A'}, Strength: {f'{strength_ab:.3f}' if strength_ab is not None else 'N/A'}")
    print(f"  B→A Accuracy: {correct_ba:.3f}, Confidence: {f'{confidence_ba:.3f}' if confidence_ba is not None else 'N/A'}, Strength: {f'{strength_ba:.3f}' if strength_ba is not None else 'N/A'}")
    print(f"  Joint Accuracy (avg): {joint_accuracy:.3f}")
    print()


# Overall accuracy statistics (averages across all pairs)
overall_ab_accuracy = sum_ab_accuracy / total_pairs
overall_ba_accuracy = sum_ba_accuracy / total_pairs
overall_joint_accuracy = sum_joint_accuracy / total_pairs

# Overall confidence statistics
overall_ab_confidence = sum_ab_confidence / count_ab_confidence if count_ab_confidence > 0 else None
overall_ba_confidence = sum_ba_confidence / count_ba_confidence if count_ba_confidence > 0 else None

# Overall strength statistics
overall_ab_strength = sum_ab_strength / count_ab_strength if count_ab_strength > 0 else None
overall_ba_strength = sum_ba_strength / count_ba_strength if count_ba_strength > 0 else None

print("=== OVERALL ACCURACY STATISTICS ===")
print(f"Total pairs processed: {total_pairs}")
print(f"\nMEAN ACCURACIES (across all pairs):")
print(f"A→B Mean Accuracy: {overall_ab_accuracy:.3f}")
print(f"B→A Mean Accuracy: {overall_ba_accuracy:.3f}")
print(f"Joint Mean Accuracy: {overall_joint_accuracy:.3f}")
print(f"\nMEAN CONFIDENCE SCORES:")
print(f"A→B Mean Confidence: {f'{overall_ab_confidence:.3f}' if overall_ab_confidence is not None else 'N/A'}")
print(f"B→A Mean Confidence: {f'{overall_ba_confidence:.3f}' if overall_ba_confidence is not None else 'N/A'}")
print(f"\nLATENCY METRICS:")
print(f"Average time per pair (both directions): {avg_latency_per_pair:.2f}s")
print(f"Average time per query: {avg_latency_per_run:.2f}s")

# Summary statistics (only valuable info, no redundant percentages)
accuracy_summary = {
    'total_pairs': total_pairs,
    'ab_mean_accuracy': overall_ab_accuracy,
    'ba_mean_accuracy': overall_ba_accuracy,
    'joint_mean_accuracy': overall_joint_accuracy,
    'ab_mean_confidence': overall_ab_confidence,
    'ba_mean_confidence': overall_ba_confidence,
    'avg_latency_per_pair_seconds': round(avg_latency_per_pair, 2),
    'avg_latency_per_query_seconds': round(avg_latency_per_run, 2),
    'total_execution_time_seconds': round(total_time, 2)
}

Pair pair0000:  Altitude ->  Temperature
  Ground Truth:  R
  A→B Accuracy: 1.000, Confidence: 0.950, Strength: 0.800
  B→A Accuracy: 1.000, Confidence: 0.950, Strength: 0.800
  Joint Accuracy (avg): 1.000

Pair pair0001:  Altitude ->  Precipitation
  Ground Truth:  R
  A→B Accuracy: 1.000, Confidence: 0.950, Strength: 0.700
  B→A Accuracy: 1.000, Confidence: 0.950, Strength: 0.700
  Joint Accuracy (avg): 1.000

Pair pair0002:  Longitude ->  Temperature
  Ground Truth:  R
  A→B Accuracy: 0.000, Confidence: 0.950, Strength: 0.000
  B→A Accuracy: 0.000, Confidence: 1.000, Strength: 0.000
  Joint Accuracy (avg): 0.000

Pair pair0003:  Altitude ->  Sunshine hours
  Ground Truth:  R
  A→B Accuracy: 1.000, Confidence: 0.800, Strength: 0.400
  B→A Accuracy: 1.000, Confidence: 0.850, Strength: 0.400
  Joint Accuracy (avg): 1.000

Pair pair0004:  Age ->  Length
  Ground Truth:  R
  A→B Accuracy: 0.000, Confidence: 0.950, Strength: 0.000
  B→A Accuracy: 1.000, Confidence: 0.950, Strength: 0.850


In [104]:
# Save accuracy results to CSV
import csv

# CSV file for detailed accuracy results (now includes confidence and strength scores)
accuracy_csv_file = "alba_accuracy_results_m1_all_pairs.csv"

# Define headers for detailed accuracy results (includes confidence and strength)
accuracy_header = [
    "PairID", "VarA", "VarB", "GroundTruth", 
    "AccuracyAB", "AccuracyBA", "JointAccuracy",
    "ConfidenceAB", "ConfidenceBA",
    "StrengthAB", "StrengthBA"
]

# Write detailed accuracy results
with open(accuracy_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=accuracy_header)
    writer.writeheader()
    for pair_id, values in accuracy_results.items():
        writer.writerow(values)

print(f"Detailed accuracy CSV file '{accuracy_csv_file}' has been created.")

# CSV file for summary statistics
summary_csv_file = "alba_accuracy_summary_m1_all_pairs.csv"

# Write summary statistics
with open(summary_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(accuracy_summary.keys()))
    writer.writeheader()
    writer.writerow(accuracy_summary)

print(f"Summary accuracy CSV file '{summary_csv_file}' has been created.")

# Display final summary table (clean format with confidence and strength)
print("\n=== FINAL SUMMARY TABLE ===")
print(f"{'Metric':<25} {'Mean Accuracy':<15} {'Mean Confidence':<18} {'Mean Strength':<15}")
print("-" * 75)
ab_conf_str = f"{accuracy_summary['ab_mean_confidence']:.3f}" if accuracy_summary['ab_mean_confidence'] is not None else 'N/A'
ba_conf_str = f"{accuracy_summary['ba_mean_confidence']:.3f}" if accuracy_summary['ba_mean_confidence'] is not None else 'N/A'

print(f"{'A→B':<25} {accuracy_summary['ab_mean_accuracy']:<15.3f} {ab_conf_str:<18} {ab_str_str:<15}")
print(f"{'B→A':<25} {accuracy_summary['ba_mean_accuracy']:<15.3f} {ba_conf_str:<18} {ba_str_str:<15}")
print(f"{'Joint Accuracy':<25} {accuracy_summary['joint_mean_accuracy']:<15.3f} {'N/A':<18} {'N/A':<15}")
print(f"{'Total Pairs':<25} {accuracy_summary['total_pairs']:<15} {'N/A':<18} {'N/A':<15}")

Detailed accuracy CSV file 'alba_accuracy_results_m1_all_pairs.csv' has been created.
Summary accuracy CSV file 'alba_accuracy_summary_m1_all_pairs.csv' has been created.

=== FINAL SUMMARY TABLE ===
Metric                    Mean Accuracy   Mean Confidence    Mean Strength  
---------------------------------------------------------------------------
A→B                       0.843           0.927              0.850          
B→A                       0.833           0.933              N/A            
Joint Accuracy            0.838           N/A                N/A            
Total Pairs               108             N/A                N/A            


save to csv test